## Silver - fact_contrato

### Transformações para a camada silver

####Importando bibliotecas

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType
from datetime import datetime



### fact_contrato
####Consultando dados da tabela bronze

In [0]:
df_contrato = spark.table("dev_credit_risk.bronze.fact_contrato")
df_contrato.limit(100).display()

#### Início das transformações
#### Retirando os espaços em branco das colunas de tipo string.

In [0]:
for field in df_contrato.schema.fields:
  if isinstance(field.dataType, StringType):
    df_contrato = df_contrato.withColumn(field.name, trim(col(field.name)))

df_contrato.display()

####Transformando colunas de data em tipo date e normalizando valores de data

In [0]:
from pyspark.sql.functions import col, try_to_date, coalesce

def corrigir_multiplas_colunas_data(df, colunas, formatos=None):
    """
    Corrige múltiplos formatos de data em várias colunas de um DataFrame
    usando try_to_date para evitar quebras de parsing no Spark moderno.
    """
    if formatos is None:
        # Coloquei o formato com barra primeiro para priorizar o seu cenário
        formatos = ["yyyy/MM/dd", "yyyy-MM-dd", "dd/MM/yyyy", "dd-MM-yyyy"]
    
    for nome_coluna in colunas:
        # Trocamos to_date por try_to_date
        tentativas = [try_to_date(col(nome_coluna), fmt) for fmt in formatos]
        df = df.withColumn(nome_coluna, coalesce(*tentativas))
        
    return df

# --- Aplicação ---
colunas_para_corrigir = ["dtContratacao", "dtLiberacao", "dtVencimentoPrimeiraParcela"]

df_contrato = corrigir_multiplas_colunas_data(df_contrato, colunas_para_corrigir)
display(df_contrato.limit(100))

####Transformando coluna valorContratado em decimal e normalizando valores

In [0]:
from pyspark.sql.functions import col, when, regexp_replace

def normalizar_coluna_para_numerico(df, nome_coluna):
    """
    Substitui os valores da coluna original, padroniza o ponto decimal
    e altera o tipo da coluna para numérico Decimal(18,2).
    """
    return df.withColumn(
        nome_coluna,
        when(col(nome_coluna).contains(","), regexp_replace(col(nome_coluna), ",", "."))
        .otherwise(col(nome_coluna))
        .cast("decimal(18,2)") # Garante o tipo numérico com 2 casas decimais fixas
    )
# --- Aplicação direta no seu DataFrame ---
df_contrato = normalizar_coluna_para_numerico(df_contrato, "valorContratado")
# Verifique que o tipo mudou de string para decimal(18,2)
df_contrato.printSchema()
# Visualizar o resultado final
display(df_contrato.limit(100))

###Padronizando os nomes das colunas

In [0]:
RENAME_MAP = {
    "idContrato": "id_contrato",
    "clienteId": "id_cliente",
    "produtoId": "id_produto",
    "scoreId": "id_score",
    "dtContratacao": "dt_contratacao",
    "dtLiberacao": "dt_liberacao",
    "dtVencimentoPrimeiraParcela": "dt_vencimento_primeira_parcela",
    "valorContratado": "valor_contratado",
    "prazoMeses": "prazo_meses"
}


####Renomeando as colunas

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df_contrato = df_contrato.withColumnRenamed(old_name, new_name)


####Exibindo o dataframe pronto para ser inserido na tabela delta silver

In [0]:
df_contrato.display()

####Inserindo os dados na tabela silver fact_contrato

In [0]:
(df_contrato.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("dev_credit_risk.silver.fact_contrato")
)

####Consultando os dados com SQL no schema tabela delta silver fact_contrato

In [0]:
%sql
select * from dev_credit_risk.silver.fact_contrato